# DubFlow - Google Colab AI service

Run all cells from top to bottom. This starts the AI API and prints the URL and
token needed by the local application.

The base install includes Whisper, SeamlessM4T, MMS, NLLB, and the `mms` voice.
Optional features are controlled in the next cell.

| Flag | Feature | Requirement |
|---|---|---|
| `MULTI_VOICE` | One Edge voice per detected speaker | Edge, pyannote and `HF_TOKEN` |
| `INSTALL_EDGE` | Edge stock voices | None |
| `INSTALL_DIARIZATION` | Speaker diarization | `HF_TOKEN` and accepted pyannote model terms |
| `INSTALL_DEMUCS` | Background separation | None |

Diarization stays unavailable until `HF_TOKEN` is configured. Models are loaded
on first use, so the first request can take longer. Colab storage is temporary;
download completed videos before the runtime stops.

After pulling new code, use **Runtime > Restart session** if an older server is
still running, then run all cells again.

In [ ]:
REPO_URL = "https://github.com/huynhphatloi/MultilingualVideoDubbingSystem.git"
BRANCH = "main"
PORT = 8000

# Used when a request does not specify an ASR model.
WHISPER_MODEL = "small"

# Required for diarization. Accept both model terms before using the token:
#   https://huggingface.co/pyannote/speaker-diarization-3.1
#   https://huggingface.co/pyannote/segmentation-3.0
HF_TOKEN = ""

# Development feature. It stays off unless this flag is true when the API starts.
MULTI_VOICE = False

# Optional packages
INSTALL_EDGE = True          # edge           - stock voice, 49 languages, no GPU
INSTALL_DIARIZATION = True   # pyannote.audio - one voice per speaker; needs HF_TOKEN
INSTALL_DEMUCS = True        # demucs         - separate speech from background


In [ ]:
from pathlib import Path

if Path("/content/dubflow").exists():
    # Refresh the existing checkout.
    !git -C /content/dubflow fetch --depth 1 origin {BRANCH} && git -C /content/dubflow reset --hard FETCH_HEAD
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/dubflow
%cd /content/dubflow/colab
!pip install -q -r requirements.txt

In [ ]:
if MULTI_VOICE and not (INSTALL_EDGE and INSTALL_DIARIZATION):
    raise ValueError("MULTI_VOICE requires INSTALL_EDGE and INSTALL_DIARIZATION")

if INSTALL_EDGE:
    !pip install -q edge-tts
if INSTALL_DIARIZATION:
    !pip install -q "pyannote.audio>=3.1"
if INSTALL_DEMUCS:
    !pip install -q demucs

print("Optional packages done. /capabilities lists what this session can load.")

In [ ]:
# Check dependency imports before starting the API.
import importlib

REQUIRED = (
    "numpy",
    "scipy",
    "numba",
    "torch",
    "transformers",
    "faster_whisper",
)

print("versions")
for name in REQUIRED:
    try:
        print(f"  {name:16}", getattr(importlib.import_module(name), "__version__", "?"))
    except Exception as exc:
        print(f"  {name:16} FAILED TO IMPORT: {type(exc).__name__}: {exc}")

# Verify the classes imported by the providers.
CHECKS = [
    ("transformers.AutoModelForSeq2SeqLM", "translation: nllb"),
    ("transformers.AutoProcessor", "translation: seamless, asr: mms"),
    ("transformers.VitsModel", "voice: mms"),
    ("faster_whisper.WhisperModel", "asr: every Whisper checkpoint"),
]
broken = []
for path, used_by in CHECKS:
    module, _, attribute = path.rpartition(".")
    try:
        getattr(importlib.import_module(module), attribute)
    except Exception as exc:
        broken.append((path, used_by, f"{type(exc).__name__}: {exc}"))

print()
if not broken:
    print("preflight OK - every pinned import resolves")
else:
    for path, used_by, reason in broken:
        print(f"BROKEN  {path}\n        needed by {used_by}\n        {reason[:300]}\n")
    print(
        "Some dependencies are incompatible. Disable the latest optional package,\n"
        "restart the runtime, and run all cells again."
    )


In [ ]:
import os
import secrets
import sys
import threading
import time
from pathlib import Path

import uvicorn

AUTH_TOKEN = secrets.token_urlsafe(24)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN
os.environ["WHISPER_MODEL"] = WHISPER_MODEL
os.environ["DUBFLOW_MULTI_VOICE"] = str(globals().get("MULTI_VOICE", False)).lower()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

# Stop the previous server before reloading project modules.
_previous_server = globals().get("api_server")
_previous_thread = globals().get("api_thread")
if _previous_server is not None:
    _previous_server.should_exit = True
    if _previous_thread is not None:
        _previous_thread.join(timeout=15)
elif _previous_thread is not None and _previous_thread.is_alive():
    raise RuntimeError(
        "A server started by an older version of this notebook is still running "
        "and cannot be stopped from here. Use Runtime > Restart session, then "
        "Run all."
    )

# Import both the Colab service and the shared project package.
_colab_dir = Path.cwd()
if not (_colab_dir / "server.py").exists():
    raise RuntimeError(
        f"Run the checkout cell first: expected the repository's colab/ folder, "
        f"but {_colab_dir} contains no server.py"
    )
for _entry in (str(_colab_dir.parent), str(_colab_dir)):
    if _entry not in sys.path:
        sys.path.insert(0, _entry)

# Reload code from the current checkout.
_project = {"server", "providers", "core", "dubflow_core"}
for _name in [n for n in list(sys.modules) if n.split(".")[0] in _project]:
    del sys.modules[_name]

import providers
import server

api_config = uvicorn.Config(server.app, host="0.0.0.0", port=PORT, log_level="warning")
api_server = uvicorn.Server(api_config)
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()
time.sleep(3)
if not api_thread.is_alive():
    raise RuntimeError("The API thread stopped immediately - see the output above")

print(f"Colab AI API v{server.app.version} started. Models load on the first request.")
for task, groups in providers.capabilities()["providers"].items():
    ready = [
        model["id"]
        for group in groups for model in group["models"] if model["available"]
    ]
    print(f"  {task:12} {len(ready):2} ready: {', '.join(ready[:6])}"
          + (" ..." if len(ready) > 6 else ""))

In [ ]:
import re
import subprocess
from pathlib import Path

cloudflared = Path("/content/cloudflared")
if not cloudflared.exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

_previous_tunnel = globals().get("tunnel")
if _previous_tunnel is not None:
    _previous_tunnel.terminate()

tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
public_url = None
for line in iter(tunnel.stdout.readline, ""):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("Cloudflare tunnel did not return a public URL")

print("Run this in the project directory on your machine:\n")
print(f"  make colab URL={public_url} TOKEN={AUTH_TOKEN}\n")
print("Or set these by hand in .env:")
print(f"  COLAB_API_URL={public_url}")
print(f"  COLAB_API_TOKEN={AUTH_TOKEN}")